# 16 · Advanced models, uncertainty & reporting

Combine models (ensemble/stacking), train on the cross-sectional order (learning-to-rank), quantify uncertainty (quantile + conformal), and assemble a full offline tearsheet / comparison dashboard.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from quantkit import backtest as B, models as MD, diagnostics as DG
from quantkit.labels import forward_return

# Synthetic cross-section with a real momentum edge (no network). On real data use
# quantkit.data.price_panel + quantkit.features; every call below is identical.
rng = np.random.default_rng(7)
dates = pd.bdate_range("2016-01-01", periods=700)
assets = [f"A{i:02d}" for i in range(25)]
mkt = rng.standard_normal((len(dates), 1)) * 0.01
prices = pd.DataFrame(
    100 * np.exp(np.cumsum(mkt + 0.008 * rng.standard_normal((len(dates), 25)), axis=0)),
    index=dates, columns=assets,
)
mom = prices / prices.shift(63) - 1.0
z = (mom - mom.rolling(126).mean()) / mom.rolling(126).std()
X, y = MD.make_design({"mom": mom, "z": z}, forward_return(prices, 5))
print("design:", X.shape, "| folds source dates:", len(dates))
from quantkit import visualization as V
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS = REPO_ROOT / 'reports'; REPORTS.mkdir(exist_ok=True)
folds = B.walk_forward(dates, train=200, test=42, horizon=5, embargo=3)

design: (12675, 2) | folds source dates: 700


## Ensembling & stacking

Each is judged on the *same* out-of-sample walk as its members — a stack is only worth it if it beats them.

In [2]:
def oos_ic(make_model):
    p = MD.walk_forward_predict(make_model, X, y, folds)
    return MD.rank_ic(p, y.loc[p.index])
print('ridge          :', round(oos_ic(MD.ridge), 4))
print('gradient_boost :', round(oos_ic(MD.gradient_boosting), 4))
print('ensemble       :', round(oos_ic(lambda: MD.EnsembleModel([MD.ridge(), MD.gradient_boosting()])), 4))
print('stacking       :', round(oos_ic(lambda: MD.StackingModel([MD.ridge, MD.gradient_boosting], MD.ridge, n_splits=4)), 4))

ridge          : 0.0147


gradient_boost : -0.0015


ensemble       : -0.0014


stacking       : -0.0265


## Learning-to-rank

Train on the within-date, zero-mean rank target — the order is what rank IC grades.

In [3]:
print('learning_to_rank:', round(oos_ic(lambda: MD.learning_to_rank(MD.ridge)), 4))

learning_to_rank: 0.0091


## Uncertainty: quantile regression + conformal intervals

In [4]:
tr = X.index.get_level_values('date') < dates[450]
qm = MD.quantile_gbm(quantiles=(0.1, 0.5, 0.9), n_estimators=80).fit(X[tr], y[tr])
print('predicted quantiles (head):')
print(qm.predict(X[~tr]).head().round(4).to_string())
cm = MD.ConformalModel(MD.ridge(1.0), alpha=0.1).fit(X[tr], y[tr])
print('\nconformal target 90% coverage -> achieved:', round(cm.coverage(X[~tr], y[~tr]), 3))

predicted quantiles (head):
                     0.1     0.5     0.9
date       asset                        
2017-09-22 A00   -0.0401 -0.0132  0.0235
           A01   -0.0288 -0.0027  0.0355
           A02   -0.0368 -0.0043  0.0216
           A03   -0.0379 -0.0043  0.0324
           A04   -0.0319 -0.0043  0.0285

conformal target 90% coverage -> achieved: 0.822


## Reporting: tearsheet, comparison dashboard, parameter explorer

Single self-contained HTML files (plotly.js embedded inline; open with no network).

In [5]:
rets_panel = prices.pct_change(fill_method=None)
w = pd.DataFrame(1.0 / 25, index=dates, columns=assets)
res = B.run_backtest(w, rets_panel, cost_model=B.CostModel(5, 2))
bench = B.buy_and_hold(rets_panel)
ts = V.tearsheet(res, REPORTS / 'nb16_tearsheet.html', benchmark=bench, title='Equal-weight tearsheet')
dash = V.comparison_dashboard({'equal_weight': res, 'buy_hold': bench}, REPORTS / 'nb16_dashboard.html')
import os
print('tearsheet :', ts.name, f'({os.path.getsize(ts)//1024} KB)')
print('dashboard :', dash.name, f'({os.path.getsize(dash)//1024} KB)')

tearsheet : nb16_tearsheet.html (4940 KB)
dashboard : nb16_dashboard.html (4886 KB)


In [6]:
# interactive parameter explorer (dropdown toggles the metric's heatmap)
grid = pd.DataFrame([[0.6, 0.8, 0.5], [0.9, 0.7, 0.4]],
                    index=pd.Index([21, 63], name='lookback'),
                    columns=pd.Index([1, 5, 21], name='skip'))
V.parameter_explorer({'sharpe': grid, 'turnover': grid * 1.5}, title='Param sweep (toy)')

**Takeaway.** Model combination and learning-to-rank target the cross-sectional order directly; conformal intervals add a coverage guarantee; the tearsheet/dashboard turn any backtest into a shareable, offline report.